In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install scikit-learn torch.utils pandas matplotlib

# PyTorch Classification Mock Model w/ Credit Data
## Goal

Build a classification model using `pytorch` upon the `Credit Risk Kaggle Dataset`. Through various steps, the data will be laundered, then put into `CreditRisk` classes. These will be put into `DataLoader`s and then interpreted by the classification model. The end goal is to determine whether or not an individual, based on given traits poses no credit risk (0) or is likely to default (1). 

## Tasks: 
1. **Setup & Data Loading:** 
    - Import `torch`, `pandas`, `sklearn`, and other dependencies.
    - Load in the dataset from `.csv`.
    - Check for cuda.

2. **Explore Data and Clean:**
    - Use `.info()`, `.describe()`, `.isnull()`, and other to find missing data/read data. 
    - Use `IterativeImputer` and/or `mean/median` to fill missing values within the dataset. 
    - Remove outliers that skew the data. i.e.: People with extensive employment lengths (60 years) or extreme age values (100 years)

3. **Feature Engineer and Preprocess Data**
    - Convert target from y/n to 1/0
    - Transform text features into numerical columns
    - Define features X and target y

4. **Data Splitting and Scaling:**
    - Split the data using `train_test_split` into training and testing sets (80/20) w/ random state (42)
    - Apply Standard Scaler to both feature sets to normalize the inputs

5. **PyTorch It!**
    - Convert numpy arrays into torch float tensors
    - Build `CreditDataset`
    - Use DataLoaders to create training and testing iterations with batch sizing

6. **Model Creation:**
    - Define your class with `nn.Linear`, `nn.ReLU`, `nn.Dropout`, and `nn.BatchNorm1d`
    - Initialize loss function. 
    - Set up Adam Optimizer with learning rate and weight decay

7. **Training Loop/Validation:**
    - Implement a training loop with the forward pass, loss calc, backprop, and weight updating
    - Check classification accuracy + loss during training
    - Check validation loss -> Check for overfitting

8. **Evaluation:**
    - Run the model in `eval()` mode. 
    - Generate performance report with Confusion matrix, Precision, recall, and f1-score

9. **Visualization + Analysis:**
    - Plot training vs validation loss over epochs
    - Visualize model mistakes through confusion heatmap

10. **Model Saving & Data Storage:**
    - Save using `torch.save()` to store the trained model weights
    - Save the `StandardScaler` using joblib to reuse it later on

## Setup & Data Loading
**Tasks:**
1. Import libraries
2. Check for Cuda availability
3. Load Credit Risk Dataset from `.csv` file within `../data/raw/`

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Check if Cuda is Available
torch.cuda.is_available()

# View Primary GPU
torch.cuda.get_device_name(0)

# Set default torch device to GPU (cuda)
torch.set_default_device('cuda')

In [ ]:
# Reading the csv file
df = pd.read_csv("../data/raw/credit_risk_dataset.csv")

# Printing Top 5 Rows
print(df.head())
print("-------------------------------------------------------------------------------------------------------")

   person_age  person_income person_home_ownership  person_emp_length  \
0          22          59000                  RENT              123.0   
1          21           9600                   OWN                5.0   
2          25           9600              MORTGAGE                1.0   
3          23          65500                  RENT                4.0   
4          24          54400                  RENT                8.0   

  loan_intent loan_grade  loan_amnt  loan_int_rate  loan_status  \
0    PERSONAL          D      35000          16.02            1   
1   EDUCATION          B       1000          11.14            0   
2     MEDICAL          C       5500          12.87            1   
3     MEDICAL          C      35000          15.23            1   
4     MEDICAL          C      35000          14.27            1   

   loan_percent_income cb_person_default_on_file  cb_person_cred_hist_length  
0                 0.59                         Y                           3  


## Explore Data and Clean
**Tasks:**
1. Use `.info()`, `.describe()`, and `.isnull()` to find missing values within the dataframe
2. Use `IterativeImputator` and `.mean(), .median()` to fill missing values if necessary
3. Check for outliers in `emp_length` and `person_age` 

In [8]:
# Shape and Size of the Data
print(df.size)
print(df.shape)

print(df.isnull().sum())

390972
(32581, 12)
person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64


In [11]:
# null values found, fix with median values
median_emp = df['person_emp_length'].median()
print(median_emp)

df['person_emp_length'] = df['person_emp_length'].fillna(median_emp)
print(df.isnull().sum())

4.0
person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length                0
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64


In [ ]:
# Thinking of iterative imputer
